In [7]:
!pip install tensorflow-text==2.9
!pip install tf-models-official==2.9.1

In [1]:
import os
import shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.interpolate as interp
from collections import OrderedDict
from ml4h.TensorMap import TensorMap, Interpretation

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from official.nlp import optimization

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

tf.get_logger().setLevel('ERROR')

2026-05-03 11:21:06.961115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777821667.207278      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777821667.278301      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777821667.888855      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777821667.888892      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777821667.888895      56 computation_placer.cc:177] computation placer alr

In [2]:
df = pd.read_csv(f'./all_split_32_v2022_05_10_meta_data.csv')
#df = pd.read_csv(f'./heroin_split_32_meta_data.csv')


step_size = 32

df = df[df.text.notna()]

df.psychoactive_class = df.psychoactive_class.apply(lambda x: x.strip())

drug2class = {d:i for i,d in enumerate(df.drug.unique())}
psychoactive2class = {d:i for i,d in enumerate(df.psychoactive_class.unique())}
ligand_chemical2class = {d:i for i,d in enumerate(df.ligand_chemical_class.unique())}
drug2weight = [(len(df)/len(df[df.drug==d])) for i,d in enumerate(df.drug.unique())]
print(psychoactive2class)
df['drug_class'] = [drug2class[d] for d in df.drug]
df['psychoactive_class_int'] = [psychoactive2class[d] for d in df.psychoactive_class]
psychoactive2weight = [(len(df)/len(df[df.psychoactive_class==d])) for d in df.psychoactive_class.unique()]
df['ligand_chemical_int'] = [ligand_chemical2class[d] for d in df.ligand_chemical_class]
chemical2weight = [(len(df)/len(df[df.ligand_chemical_class==d])) for d in df.ligand_chemical_class.unique()]
df[[f'tag_{i}' for i in range(52) ]] = df[[f'tag_{i}' for i in range(52) ]].fillna(0)
df[[f'tag_{i}' for i in range(52) ]] = df[[f'tag_{i}' for i in range(52) ]].astype(int)
tag2weight = {t:len(df)/sum(df[t]) for t in [f'tag_{i}' for i in range(52)] if sum(df[t])>100}

# train = df.sample(frac = 0.8)
# test = df.drop(train.index).sample(frac = 0.5)
# validate = df.drop(train.index).drop(test.index)

# df[df.set=='train'].to_csv('train.csv', index=False)
# df[df.set=='valid'].to_csv('valid.csv', index=False)
# df[df.set=='test'].to_csv('test.csv', index=False)

tags = {'Small_Group': 'tag_0', 'General': 'tag_1', 'First_Times': 'tag_2', 'Alone': 'tag_3', 'Difficult_Experiences': 'tag_4', 'Glowing_Experiences': 'tag_5', 'Retrospective_Summary': 'tag_6', 'Various': 'tag_7', 'Unknown_Context': 'tag_8', 'Mystical_Experiences': 'tag_9', 'Health_Problems': 'tag_10', 'Combinations': 'tag_11', 'Not_Applicable': 'tag_12', 'Bad_Trips': 'tag_13', 'Hangover_Days_After': 'tag_14', 'Entities_Beings': 'tag_15', 'Music_Discussion': 'tag_16', 'Addiction_Habituation': 'tag_17', 'Post_Trip_Problems': 'tag_18', 'Nature_Outdoors': 'tag_19', 'Relationships': 'tag_20', 'Depression': 'tag_21', 'Therapeutic_Intent_or_Outcome': 'tag_22', 'Overdose': 'tag_23', 'Medical_Use': 'tag_24', 'Sex_Discussion': 'tag_25', 'Train_Wrecks_Trip_Disasters': 'tag_26', 'Guides_Sitters': 'tag_27', 'Rave_Dance_Event': 'tag_28', 'Preparation_Recipes': 'tag_29', 'Festival_Lg_Crowd': 'tag_30', 'Health_Benefits': 'tag_31', 'Large_Group': 'tag_32', 'Multi-Day_Experience': 'tag_33', 'Club_Bar': 'tag_34', 'What_Was_in_That': 'tag_35', 'Personal_Preparation': 'tag_36', 'HPPD_Lasting_Visuals': 'tag_37', 'Families': 'tag_38', 'Second_Hand_Report': 'tag_39', 'Loss_of_Magic': 'tag_40', 'Hospital': 'tag_41', 'Public_Space': 'tag_42', 'School': 'tag_43', 'Poetry': 'tag_44', 'Performance_Enhancement': 'tag_45', 'Large_Party': 'tag_46', 'Group_Ceremony': 'tag_47', 'Workplace': 'tag_48', 'Cultivation_Synthesis': 'tag_49', 'Pregnancy_Baby': 'tag_50', 'Military': 'tag_51'}
itags = {v: k.replace('(', '').replace(')', '').replace('/', '') for k, v in tags.items()}
ctags = {v: int(k.replace('tag_', '')) for k, v in itags.items()}

receptors = ['5_ht2a', '5_ht2c', '5_ht2b', '5_ht1a', '5_ht1b', '5_ht1d', '5_ht1e', '5_ht1f', '5_ht3', '5_ht5a', '5_ht6', '5_ht7', 'dopamine_d1', 'dopamine_d2', 'dopamine_d3', 'dopamine_d4', 'dopamine_d5', 'adrenergic_alpha1a', 'adrenergic_alpha1b', 'adrenergic_alpha2a', 'adrenergic_alpha2b', 'adrenergic_beta1', 'adrenergic_beta2', 'sert', 'dat', 'net', 'imidazoline_1', 'sigma_1', 'sigma_2', 'dor', 'kor', 'mor', 'm1', 'm2', 'm3', 'm4', 'm5', 'h1', 'h2', 'h3', 'h4', 'calcium_channel', 'nmda', 'cb1', 'cb2', 'glutamate_ampa', 'gaba_a', 'gaba_b', 'dopamine_d2_long', 'dopamine_d2_short', 'sodium_channel', 'taar1', 'substance_p', 'paf_platelet_activating_factor', 'prostaglandin_e3', 'prostaglandin_e4', 'herg', 'monoamine_oxidase_a', 'monoamine_oxidase_b', 'cholecystokinin_a', 'cholecystokinin_b']
df[receptors] = df[receptors].astype(float)

{'psychedelic': 0, 'dissociative': 1, 'entactogen': 2, 'hallucinogen': 3, 'depressant': 4, 'stimulant': 5, 'antipsychotic': 6, 'antidepressant': 7, 'deliriant': 8, 'opioid': 9}


In [3]:
output_cols = [ 'psychoactive_class_int', 'ligand_chemical_int', 'drug_class']
output_cols += [f'tag_{i}' for i in range(52) if sum(df[f'tag_{i}']) > 100]


weighted_loss = False
tensor_maps_out = []
scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def weight_scce(weight_array):
    class_weights = tf.constant(weight_array)
    def my_loss(y_true, y_pred):
        weights = tf.gather(class_weights, y_true)
        return tf.compat.v1.losses.sparse_softmax_cross_entropy(y_true, y_pred, weights)
    return my_loss


def pearson(y_true, y_pred):
    # normalizing stage - setting a 0 mean.
    y_true -= K.mean(y_true, axis=-1)
    y_pred -= K.mean(y_pred, axis=-1)
    # normalizing stage - setting a 1 variance
    y_true = K.l2_normalize(y_true, axis=-1)
    y_pred = K.l2_normalize(y_pred, axis=-1)
    # final result
    pearson_correlation = K.sum(y_true * y_pred, axis=-1)
    return pearson_correlation

for oc in output_cols:
    if 'drug_class' == oc:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CATEGORICAL, shape=(1,),
                                         loss=weight_scce(drug2weight) if weighted_loss else scce,
                                         metrics=[tf.metrics.SparseCategoricalAccuracy()],
                                         channel_map={f'drug_{d}': v for d,v in drug2class.items()}))
    elif 'psychoactive_class_int' == oc:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CATEGORICAL, shape=(1,),
                                         loss=weight_scce(psychoactive2weight) if weighted_loss else scce,
                                         metrics=[tf.metrics.SparseCategoricalAccuracy()],
                                         channel_map={f'{d}': v for d,v in psychoactive2class.items()}))
    elif 'ligand_chemical_int' == oc:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CATEGORICAL, shape=(1,),
                                         loss=weight_scce(chemical2weight) if weighted_loss else scce,
                                         metrics=[tf.metrics.SparseCategoricalAccuracy()],
                                         channel_map={f'{d}': v for d,v in ligand_chemical2class.items()}))        
    elif 'tag_' in oc:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CATEGORICAL, shape=(1,), 
                                         loss=weight_scce([1.1, tag2weight[oc]]) if weighted_loss else scce,
                                         metrics=[tf.metrics.SparseCategoricalAccuracy()],
                                         channel_map={f'no_{itags[oc]}': 0, f'{itags[oc]}': 1}))
    elif 'age' in oc:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CONTINUOUS, shape=(1,),
                                         loss=tf.keras.losses.MeanSquaredError(),
                                         metrics=[pearson]
                                         
                                        ))
    elif oc in receptors:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CONTINUOUS, shape=(1,),
                                         loss=tf.keras.losses.LogCosh(),
                                         metrics=[tf.metrics.MeanAbsoluteError()],))
    else:
        tensor_maps_out.append(TensorMap(f'{oc}', Interpretation.CATEGORICAL, shape=(1,), 
                                         loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                                         metrics=[tf.metrics.SparseCategoricalAccuracy()],
                                         channel_map={f'no_{oc}':0, f'{oc}':1}))


I0000 00:00:1777821761.971426      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14785 MB memory:  -> device: 0, name: Tesla V100-SXM2-16GB, pci bus id: 0000:00:04.0, compute capability: 7.0


In [17]:
import keras
epochs = 5
num_train_steps = 100 * epochs
num_warmup_steps = int(0.1*num_train_steps)

init_lr = 3e-5
optimizer = optimization.create_optimizer(init_lr=init_lr,
                                          num_train_steps=num_train_steps,
                                          num_warmup_steps=num_warmup_steps,
                                          optimizer_type='adamw')
saved_model_path = "./models/bertowid_categorical_v2022_08_03"

import tensorflow as tf
import keras


class BertowidClassifierModel(keras.Model):
    def __init__(self, saved_model_path, output_key=None, **kwargs):
        super().__init__(**kwargs)

        # Load and run this model on CPU because the SavedModel has
        # CPU-bound lookup-table resources for vocab/text preprocessing.
        with tf.device("/CPU:0"):
            self.saved = tf.saved_model.load(saved_model_path)
            self.fn = self.saved.signatures["serving_default"]

        self.output_key = output_key

    def call(self, inputs, training=False):
        with tf.device("/CPU:0"):
            outputs = self.fn(text=inputs)

        if self.output_key is not None:
            return outputs[self.output_key]

        return outputs

In [19]:
import tensorflow as tf


saved_model_path = "./models/bertowid_categorical_v2022_08_03"

with tf.device("/CPU:0"):
    saved = tf.saved_model.load(saved_model_path)
    infer = saved.signatures["serving_default"]

texts = tf.constant([
    "example sentence one",
    "example sentence two",
])

with tf.device("/CPU:0"):
    preds = infer(text=texts)

print(preds.keys())

dict_keys(['tag_24', 'tag_7', 'tag_43', 'tag_27', 'tag_33', 'tag_47', 'drug_class', 'tag_14', 'tag_23', 'tag_3', 'tag_51', 'tag_17', 'tag_48', 'tag_20', 'tag_8', 'tag_16', 'tag_22', 'tag_31', 'tag_36', 'tag_26', 'tag_34', 'tag_21', 'tag_41', 'tag_28', 'tag_4', 'tag_2', 'tag_46', 'tag_18', 'tag_29', 'tag_38', 'tag_39', 'tag_40', 'tag_45', 'tag_15', 'tag_42', 'tag_0', 'tag_11', 'tag_9', 'tag_30', 'tag_12', 'tag_1', 'tag_10', 'tag_13', 'tag_5', 'tag_6', 'ligand_chemical_int', 'tag_50', 'tag_19', 'tag_32', 'tag_49', 'tag_37', 'psychoactive_class_int', 'tag_25', 'tag_35', 'tag_44'])


In [25]:
words = tf.constant([
    'It was infuriating how they never listen to me',
    'I learned a lot I had never seen so deeply into the heart of god and the universe.',
    'the desire is strong to repeat the experience on both smaller doses',
    'with music available for contemplation - at the beach - at night.',
    'There is complete recall of all experiences and thoughts, after the drug has been dissipated.',
    'The feeling of comfort, introspection, and tranquility',
    'I danced and raved all night long'
])

# New classifier_model returns a dict of output heads.
with tf.device("/CPU:0"):
    r = infer(text=words)

int2drug = {i: d for i, d in enumerate(df.drug.unique())}
int2psychoactive = {i: d for i, d in enumerate(df.psychoactive_class.unique())}
int2chemical = {i: d for i, d in enumerate(df.ligand_chemical_class.unique())}

# Main class guesses
drug_guess = tf.argmax(r["drug_class"], axis=-1).numpy()
psychoactive_guess = tf.argmax(r["psychoactive_class_int"], axis=-1).numpy()
chemical_guess = tf.argmax(r["ligand_chemical_int"], axis=-1).numpy()

print("drug guesses:", drug_guess)

# Stable output ordering, replacing classifier_model.output_names
output_names = list(r.keys())

# Optional: put major heads first, then tags
major_heads = [
    "drug_class",
    "psychoactive_class_int",
    "ligand_chemical_int",
]

tag_heads = sorted(
    [k for k in output_names if k.startswith("tag_")],
    key=lambda x: int(x.split("_")[1])
)

ordered_output_names = major_heads + tag_heads

for j, s in enumerate(words):
    print()
    print(s)

    for ot in ordered_output_names:
        if ot not in r:
            continue

        logits_or_probs = r[ot]

        if ot.startswith("tag_"):
            # Each tag head has shape: (n_examples, 2)
            # Class 1 is the positive class.
            score = tf.nn.softmax(logits_or_probs[j], axis=-1)[1].numpy()

            # itags may be keyed by "tag_0", "tag_1", etc.
            tag_name = itags.get(ot, ot)

            print(f'  {tag_name}: {score:0.2f}')

        else:
            class_guess = tf.argmax(logits_or_probs[j], axis=-1).numpy()

            if ot == "drug_class":
                print(f'  drug guess: {class_guess} is {int2drug[class_guess]}')

            elif ot == "psychoactive_class_int":
                print(
                    f'  Psychoactive guess: '
                    f'{class_guess} is {int2psychoactive[class_guess]}'
                )

            elif ot == "ligand_chemical_int":
                print(
                    f'  Chemical guess: '
                    f'{class_guess} is {int2chemical[class_guess]}'
                )

drug guesses: [50 22 14 25 10 24 24]

tf.Tensor(b'It was infuriating how they never listen to me', shape=(), dtype=string)
  drug guess: 50 is thc
  Psychoactive guess: 5 is stimulant
  Chemical guess: 0 is phenethylamine
  Small_Group: 0.29
  General: 0.27
  First_Times: 0.25
  Alone: 0.11
  Difficult_Experiences: 0.34
  Glowing_Experiences: 0.08
  Retrospective_Summary: 0.27
  Various: 0.26
  Unknown_Context: 0.03
  Mystical_Experiences: 0.03
  Health_Problems: 0.08
  Combinations: 0.04
  Not_Applicable: 0.12
  Bad_Trips: 0.14
  Hangover_Days_After: 0.15
  Entities_Beings: 0.05
  Music_Discussion: 0.03
  Addiction_Habituation: 0.12
  Post_Trip_Problems: 0.09
  Nature_Outdoors: 0.02
  Relationships: 0.07
  Depression: 0.11
  Therapeutic_Intent_or_Outcome: 0.02
  Overdose: 0.04
  Medical_Use: 0.03
  Sex_Discussion: 0.02
  Train_Wrecks_Trip_Disasters: 0.06
  Guides_Sitters: 0.01
  Rave_Dance_Event: 0.01
  Preparation_Recipes: 0.00
  Festival_Lg_Crowd: 0.02
  Health_Benefits: 0.01
  Larg

In [ ]:
words = ['It was infuriating how they never listen to me',
         'I learned a lot I had never seen so deeply into the heart of god and the universe.', 
         'the desire is strong to repeat the experience on both smaller doses' ,
         'with music available for contemplation - at the beach - at night.',
         'There is complete recall of all experiences and thoughts, after the drug has been dissipated.',
         'The feeling of comfort, introspection, and tranquility',
         'I danced and raved all night long'
        ]

In [15]:
int2drug = {i:d for i,d in enumerate(df.drug.unique())}
int2psychoactive = {i:d for i,d in enumerate(df.psychoactive_class.unique())}
int2chemical = {i:d for i,d in enumerate(df.ligand_chemical_class.unique())}

In [16]:
for j,s in enumerate(words):
    for i,ot in enumerate(classifier_model.output_names):
        if 'tag' in ot:
            score = sigmoids[i-3, j, 1].numpy()
            score2 = tf.nn.softmax(r[i][j]).numpy()
            print(f'{s[:60]} <- {itags[ot]}: {score:0.2f}:')
        else:
            drug_guess = tf.argmax(r[i][j], -1).numpy()
            if r[i].shape[-1] == len(int2drug):
                print(f'drug guess: {drug_guess} is {int2drug[drug_guess]}')
            elif r[i].shape[-1] == len(int2psychoactive):
                print(f'Psychoactive guess: {drug_guess} is {int2psychoactive[drug_guess]}')
            elif r[i].shape[-1] == len(int2chemical):
                print(f'Chemical guess is {int2chemical[drug_guess]}')                

Psychoactive guess: 5 is stimulant
Chemical guess is phenethylamine
drug guess: 50 is thc
It was infuriating how they never listen to me <- Small_Group: 0.29:
It was infuriating how they never listen to me <- General: 0.27:
It was infuriating how they never listen to me <- First_Times: 0.25:
It was infuriating how they never listen to me <- Alone: 0.11:
It was infuriating how they never listen to me <- Difficult_Experiences: 0.34:
It was infuriating how they never listen to me <- Glowing_Experiences: 0.08:
It was infuriating how they never listen to me <- Retrospective_Summary: 0.27:
It was infuriating how they never listen to me <- Various: 0.26:
It was infuriating how they never listen to me <- Unknown_Context: 0.03:
It was infuriating how they never listen to me <- Mystical_Experiences: 0.03:
It was infuriating how they never listen to me <- Health_Problems: 0.08:
It was infuriating how they never listen to me <- Combinations: 0.04:
It was infuriating how they never listen to me <- 

The feeling of comfort, introspection, and tranquility <- Addiction_Habituation: 0.03:
The feeling of comfort, introspection, and tranquility <- Post_Trip_Problems: 0.02:
The feeling of comfort, introspection, and tranquility <- Nature_Outdoors: 0.03:
The feeling of comfort, introspection, and tranquility <- Relationships: 0.04:
The feeling of comfort, introspection, and tranquility <- Depression: 0.04:
The feeling of comfort, introspection, and tranquility <- Therapeutic_Intent_or_Outcome: 0.09:
The feeling of comfort, introspection, and tranquility <- Overdose: 0.01:
The feeling of comfort, introspection, and tranquility <- Medical_Use: 0.03:
The feeling of comfort, introspection, and tranquility <- Sex_Discussion: 0.02:
The feeling of comfort, introspection, and tranquility <- Train_Wrecks_Trip_Disasters: 0.01:
The feeling of comfort, introspection, and tranquility <- Guides_Sitters: 0.02:
The feeling of comfort, introspection, and tranquility <- Rave_Dance_Event: 0.01:
The feeling 

In [26]:
reddit = pd.read_csv('./SEPT_acute_filtered.tsv', sep='\t')
reddit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26658 entries, 0 to 26657
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Drug Type        26658 non-null  object
 1   Timing           26658 non-null  object
 2   Trip Report      26658 non-null  object
 3   Multiple Drugs   26658 non-null  object
 4   Previous Trips   26658 non-null  object
 5   Trip             26658 non-null  object
 6   subreddit        26658 non-null  object
 7   author           26658 non-null  object
 8   created_utc      26658 non-null  object
 9   Time adjustment  26658 non-null  int64 
 10  Adjusted Timing  26658 non-null  object
dtypes: int64(1), object(10)
memory usage: 2.2+ MB


In [47]:
from collections import defaultdict
import numpy as np
import tensorflow as tf

scores = defaultdict(list)

texts = reddit["Trip"].astype(str).to_numpy()
batch_size = 16

for start in range(0, len(texts), batch_size):
    batch_texts = texts[start:start + batch_size]

    with tf.device("/CPU:0"):
        r = infer(text=tf.constant(batch_texts))

    for ot in ordered_output_names:
        if not ot.startswith("tag_") or ot not in r:
            continue

        # Shape: (batch_size, 2), take positive-class probability.
        tag_scores = tf.nn.softmax(r[ot], axis=-1)[:, 1].numpy()

        tag_name = itags.get(ot, ot)
        scores[tag_name].extend(tag_scores.tolist())

    count = min(start + batch_size, len(texts))
    if count % 50 == 0 or count == len(texts):
        print(f'Processed {count} testimonials')
    

Processed 400 testimonials
Processed 800 testimonials
Processed 1200 testimonials
Processed 1600 testimonials
Processed 2000 testimonials
Processed 2400 testimonials
Processed 2800 testimonials
Processed 3200 testimonials
Processed 3600 testimonials
Processed 4000 testimonials
Processed 4400 testimonials
Processed 4800 testimonials
Processed 5200 testimonials
Processed 5600 testimonials
Processed 6000 testimonials
Processed 6400 testimonials
Processed 6800 testimonials
Processed 7200 testimonials
Processed 7600 testimonials
Processed 8000 testimonials
Processed 8400 testimonials
Processed 8800 testimonials
Processed 9200 testimonials
Processed 9600 testimonials
Processed 10000 testimonials
Processed 10400 testimonials
Processed 10800 testimonials
Processed 11200 testimonials
Processed 11600 testimonials
Processed 12000 testimonials
Processed 12400 testimonials
Processed 12800 testimonials
Processed 13200 testimonials
Processed 13600 testimonials
Processed 14000 testimonials
Processed 1

In [48]:
scores_df = pd.DataFrame(scores)

In [49]:
scores_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26658 entries, 0 to 26657
Data columns (total 52 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Small_Group                    26658 non-null  float64
 1   General                        26658 non-null  float64
 2   First_Times                    26658 non-null  float64
 3   Alone                          26658 non-null  float64
 4   Difficult_Experiences          26658 non-null  float64
 5   Glowing_Experiences            26658 non-null  float64
 6   Retrospective_Summary          26658 non-null  float64
 7   Various                        26658 non-null  float64
 8   Unknown_Context                26658 non-null  float64
 9   Mystical_Experiences           26658 non-null  float64
 10  Health_Problems                26658 non-null  float64
 11  Combinations                   26658 non-null  float64
 12  Not_Applicable                 26658 non-null 

In [50]:
red52 = reddit #.head(1000)

In [51]:
red52.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26658 entries, 0 to 26657
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Drug Type        26658 non-null  object
 1   Timing           26658 non-null  object
 2   Trip Report      26658 non-null  object
 3   Multiple Drugs   26658 non-null  object
 4   Previous Trips   26658 non-null  object
 5   Trip             26658 non-null  object
 6   subreddit        26658 non-null  object
 7   author           26658 non-null  object
 8   created_utc      26658 non-null  object
 9   Time adjustment  26658 non-null  int64 
 10  Adjusted Timing  26658 non-null  object
dtypes: int64(1), object(10)
memory usage: 2.2+ MB


In [52]:
red52 = red52.join(scores_df, how='inner')
red52.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26658 entries, 0 to 26657
Data columns (total 63 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Drug Type                      26658 non-null  object 
 1   Timing                         26658 non-null  object 
 2   Trip Report                    26658 non-null  object 
 3   Multiple Drugs                 26658 non-null  object 
 4   Previous Trips                 26658 non-null  object 
 5   Trip                           26658 non-null  object 
 6   subreddit                      26658 non-null  object 
 7   author                         26658 non-null  object 
 8   created_utc                    26658 non-null  object 
 9   Time adjustment                26658 non-null  int64  
 10  Adjusted Timing                26658 non-null  object 
 11  Small_Group                    26658 non-null  float64
 12  General                        26658 non-null 

In [53]:
sorted_df = red52.sort_values(by='Mystical_Experiences')

print(sorted_df[['Mystical_Experiences', 'Trip']].tail())

       Mystical_Experiences                                               Trip
14564              0.683846  Oneness and State of Being Two weeks ago, I ha...
17489              0.687185  I just met mother (god?) After dipping my toes...
18806              0.687372  i wrote this right after my second trip, a few...
4806               0.717662  My Experience with DMT, this is no drug... My ...
3974               0.734821  Break through? I don’t feel like it’s a questi...


In [54]:
red52.to_csv('trips_bertowid_predictions_v2026_05_03.tsv', sep='\t', index=False)

In [2]:
!pip install wordcloud

In [1]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Assuming red52 is your DataFrame and it has a column named 'Trip'
# Combine all text from the 'Trip' column into a single string.
# We use dropna() to avoid any potential None/NaN values.
text = " ".join(str(word) for word in red52['Trip'].dropna())

# Create a WordCloud object. Customize parameters like width, height, and background color as needed.
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

# Display the word cloud using matplotlib
plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # Hide axes for better display
plt.show()


ModuleNotFoundError: No module named 'wordcloud'

In [16]:
[s for s in scores]

['25i-nbome',
 '2c-b',
 '2c-c',
 '2c-d',
 '2c-e',
 '2c-i',
 '2c-p',
 '2c-t-2',
 '2c-t-4',
 '2c-t-7',
 '5-meo-dmt',
 '5-meo-dipt',
 '5-meo-mipt',
 '5-meo-tmt',
 'dmt',
 'dob',
 'doi',
 'dom',
 'dpt',
 'dxm',
 'dipt',
 'ketamine',
 'lsd',
 'mda',
 'mdma',
 'mescaline',
 'pcp',
 'psilocin',
 'salvia',
 'tma-2',
 'alprazolam',
 'amphetamine',
 'aripiprazole',
 'bupropion',
 'cocaine',
 'diphenhydramine',
 'haloperidol',
 'hydrocodone',
 'hydromorphone',
 'ibogaine',
 'methadone',
 'methamphetamine',
 'mirtazapine',
 'morphine',
 'olanzapine',
 'oxycodone',
 'paroxetine',
 'quetiapine',
 'risperidone',
 'sertraline',
 'thc',
 'venlafaxine']

In [ ]:
standardized = rec_dd()
peaks=Counter()
peak_index = {}
peak_testimonials={}
new_size = 30
for drug in scores:
    for t in scores[drug]:
        for si, s in enumerate(scores[drug][t]):
            try:
                interpolator = interp.interp1d(np.arange(len(scores[drug][t][s])), scores[drug][t][s])
                standardized[drug][t][s] = interpolator(np.linspace(0, len(scores[drug][t][s])-1, new_size))
                if max(scores[drug][t][s]) > peaks[drug,s]:
                    peak_testimonials[drug,s] = t
                    peaks[drug,s] = max(scores[drug][t][s])
                    peak_index[drug,s] = np.argmax(scores[drug][t][s])
            except ValueError as e:
                print(f'vall err {drug} {t} {e}')

In [ ]:
for d in scores:
    for t in scores[d]:
        for p in scores[d][t]:
            print(f' {d} {t} {p} {scores[d][t][p]}')
            break

In [ ]:
print(tags)

In [ ]:
pzz = []
count = []
testi = []
peak_sentence = {}
for p in peaks:
    
    if p[-1] == tags['Therapeutic_Intent_or_Outcome']:
        #print(p)
        #print(f'{p} has: {peaks[p]:0.3f} and testi: {peak_testimonials[p]} pi {peak_index[p]}')
        count.append(peaks[p])
        testi.append(peak_testimonials[p])

        pzz.append(p)
print(f'max has: {max(count)} and testi: {testi[np.argmax(count)]} at sentence: {peak_index[pzz[np.argmax(count)]]} drug, feeling {pzz[np.argmax(count)]}')
peak_key = pzz[np.argmax(count)]
ex = df[df.drug == peak_key[0]]
ex = ex[ex.testimonial == testi[np.argmax(count)]]
ex = ex[ex.sentence==peak_index[peak_key]]
print(ex.iloc[0].text)

In [ ]:
peak_key = pzz[np.argmax(count)]
ex = df[df.drug == peak_key[0]]
ex = ex[ex.testimonial == testi[np.argmax(count)]]
ex = ex[ex.sentence==peak_index[peak_key]]
print(ex.iloc[0].text)

In [ ]:
standardized = rec_dd()
peaks=Counter()
peak_index = {}
peak_testimonials={}

mins=Counter()
mins_index = {}
mins_testimonials={}
new_size = 25
for drug in scores:
    for t in scores[drug]:
        for si, s in enumerate(scores[drug][t]):
            try:
                interpolator = interp.interp1d(np.arange(len(scores[drug][t][s])), scores[drug][t][s])
                standardized[drug][t][s] = interpolator(np.linspace(0, len(scores[drug][t][s])-1, new_size))
                if max(scores[drug][t][s]) > peaks[drug,s]:
                    peak_testimonials[drug,s] = t
                    peaks[drug,s] = max(scores[drug][t][s])
                    peak_index[drug,s] =  np.argmax(scores[drug][t][s])
                
                if mins[drug,s] == 0:
                    mins[drug,s] = 1
                if min(scores[drug][t][s]) < mins[drug,s]:
                    mins_testimonials[drug,s] = t
                    mins[drug,s] = min(scores[drug][t][s])
                    mins_index[drug,s] = np.argmin(scores[drug][t][s])
            except ValueError:
                print(f'vall err {drug} {t}')
                break

In [ ]:
data = defaultdict(list)
for sentiment in scores[drug][t]:
    s = itags[sentiment] if 'tag_' in sentiment else sentiment
    if 'tag' not in sentiment:
        continue
    for p in peaks:
        if sentiment in p:

            print(f'\n{s.capitalize()} at drug {p[0]} Testimonial: {peak_testimonials[p]} Sentence: {peak_index[p]}')
            peak_key = p # pzz[np.argmax(count)]
            ex = df[df.drug == peak_key[0]]
            ex = ex[ex.testimonial == peak_testimonials[p]]
            ex = ex[ex.sentence==peak_index[peak_key]]
            print(ex.iloc[0].text)
            data['tag'].append(s)
            data['drug'].append(p[0])
            data['psychoactive_class'].append(ex.iloc[0].psychoactive_class)
            data['ligand_chemical_class'].append(ex.iloc[0].ligand_chemical_class)
            data['text'].append(ex.iloc[0].text)
            data['score'].append(peaks[p])
            data['testimonial'].append(peak_testimonials[p])
            data['sentence'].append(peak_index[p])
            data['tag_number'].append(sentiment)
    for p in mins:
        if sentiment in p:

            print(f'\nNOT {s.capitalize()} at drug {p[0]} Testimonial: {mins_testimonials[p]} Sentence: {mins_index[p]}')
            ex = df[df.drug == p[0]]
            ex = ex[ex.testimonial == mins_testimonials[p]]
            ex = ex[ex.sentence==mins_index[p]]
            print(ex.iloc[0].text)
            data['tag'].append('Not_'+s)
            data['drug'].append(p[0])
            data['psychoactive_class'].append(ex.iloc[0].psychoactive_class)
            data['ligand_chemical_class'].append(ex.iloc[0].ligand_chemical_class)
            data['text'].append(ex.iloc[0].text)
            data['score'].append(mins[p])
            data['testimonial'].append(mins_testimonials[p])
            data['sentence'].append(mins_index[p])
            data['tag_number'].append(sentiment)
                
    print()

In [ ]:
exdf = pd.DataFrame.from_dict(data)
exdf.to_csv('most_extreme_erowid_tags_per_drug.csv', index=False)

In [ ]:
data = defaultdict(list)
for sentiment in scores[drug][t]:
    s = itags[sentiment] if 'tag_' in sentiment else sentiment
    if 'tag' not in sentiment:
        continue
    count = []
    testi = []
    pzz = []
    for p in peaks:
        if sentiment in p:
            count.append(peaks[p])
            testi.append(peak_testimonials[p])
            pzz.append(p)
    print(f'\nmax {s.capitalize()} at drug {pzz[np.argmax(count)][0]} Testimonial: {testi[np.argmax(count)]} Sentence: {peak_index[pzz[np.argmax(count)]]}')
    print(f'\nmax Not {s.capitalize()} at drug {pzz[np.argmax(count)][0]} Testimonial: {testi[np.argmax(count)]} Sentence: {peak_index[pzz[np.argmax(count)]]}')
    
    peak_key = pzz[np.argmax(count)]
    ex = df[df.drug == peak_key[0]]
    ex = ex[ex.testimonial == peak_testimonials[peak_key]]
    ex = ex[ex.sentence == peak_index[peak_key]]
    print(ex.iloc[0].text)
    print()
    data['tag'].append(s)
    
    data['drug'].append(peak_key[0])
    data['psychoactive_class'].append(ex.iloc[0].psychoactive_class)
    data['ligand_chemical_class'].append(ex.iloc[0].ligand_chemical_class)
    data['text'].append(ex.iloc[0].text)
    data['score'].append(peaks[peak_key])
    data['testimonial'].append(peak_testimonials[peak_key])
    data['sentence'].append(peak_index[peak_key])
    data['tag_number'].append(sentiment)
    
    count = []
    testi = []
    pzz = []
    for p in mins:
        if sentiment in p:
            count.append(mins[p])
            testi.append(mins_testimonials[p])
            pzz.append(p)
            
    print(f'\nmax Not {s.capitalize()} at drug {pzz[np.argmin(count)][0]} Testimonial: {testi[np.argmin(count)]} Sentence: {peak_index[pzz[np.argmax(count)]]}')
    
    min_key = pzz[np.argmin(count)]
    ex = df[df.drug == min_key[0]]
    ex = ex[ex.testimonial == mins_testimonials[min_key]]
    ex = ex[ex.sentence==mins_index[min_key]]
    print(ex.iloc[0].text)
    print()
    data['tag'].append("Not_"+s)
    data['drug'].append(min_key[0])
    data['psychoactive_class'].append(ex.iloc[0].psychoactive_class)
    data['ligand_chemical_class'].append(ex.iloc[0].ligand_chemical_class)
    data['text'].append(ex.iloc[0].text)
    data['score'].append(mins[min_key])
    data['testimonial'].append(mins_testimonials[min_key])
    data['sentence'].append(mins_index[min_key])
    data['tag_number'].append(sentiment)
    
exdf = pd.DataFrame.from_dict(data)
exdf.to_csv('most_extreme_erowid_tags.csv', index=False)

In [ ]:
standardized = rec_dd()
means = Counter()
stds = Counter()
new_size = 30
for drug in scores:
    for t in scores[drug]:
        for s in scores[drug][t]:
            try:
                interpolator = interp.interp1d(np.arange(len(scores[drug][t][s])), scores[drug][t][s])
                standardized[drug][t][s] = interpolator(np.linspace(0, len(scores[drug][t][s])-1, new_size))
                means[drug,s] += standardized[drug][t][s]
                stds[drug,s] += standardized[drug][t][s]*standardized[drug][t][s]
            except ValueError:
                print(f'vall err {drug} {t}')
                break
    for s in scores[drug][t]:
        means[drug,s] /= len(scores[drug])
        stds[drug,s] /= len(scores[drug])
        stds[drug,s] -= means[drug,s]*means[drug,s]
        stds[drug,s] = np.sqrt(stds[drug,s])

In [17]:
def get_mean_trajectories(scores, column='psychoactive_class', new_size = 25):
    standardized_psycho = rec_dd()
    means_psycho = Counter()
    stds_psycho = Counter()
    
    print(output_cols)
    for drug in scores:
        psychoactive_class = df[df.drug==drug][column].unique()[0]
        for t in scores[drug]:
            for s in scores[drug][t]:
                try:
                    interpolator = interp.interp1d(np.arange(len(scores[drug][t][s])), scores[drug][t][s])

                    standardized_psycho[drug][t][s] = interpolator(np.linspace(0, len(scores[drug][t][s])-1, new_size))
                    means_psycho[psychoactive_class,s] += standardized_psycho[drug][t][s]
                    stds_psycho[psychoactive_class,s] += standardized_psycho[drug][t][s]*standardized_psycho[drug][t][s]
                except ValueError as e:
                    print(f'vall err {drug} {t} len: {len(scores[drug][t][s])} e:{e}')
                    break

    for pc in df[column].unique():
        n = 1
        for drug in df[df[column]==pc].drug.unique():
            n += len(standardized_psycho[drug])
        print(f'Class {pc} has length: {n}')
        for s in output_cols:
            means_psycho[pc,s] /= n
            stds_psycho[pc,s] /= n
            stds_psycho[pc,s] -= means_psycho[pc,s]*means_psycho[pc,s]
            stds_psycho[pc,s] = np.sqrt(stds_psycho[pc,s])
    return standardized_psycho, means_psycho, stds_psycho

def plot_tag_trajectories(labels, means, stds, keep_drugs, keep_tags,
                   figsize=(42, 42), cols=5, rows=4, dpi=300, ncol=4):
    fig, axes = plt.subplots(rows, cols, figsize=figsize, dpi=dpi)
    cmap = plt.get_cmap('hsv')
    colors = cmap(np.linspace(0, 1, len(labels)))
    markers = "v.,o1^2>348<spdxh*PH+XD|"
    o_drugs = sorted(labels)
    o_tags = [ot for ot in classifier_model.output_names if 'tag_' in ot and itags[ot] in keep_tags]

    for sentiment, ax in zip(o_tags, axes.ravel()):
        for i, (drug, color) in enumerate(zip(o_drugs, colors)):
            if drug not in keep_drugs:
                continue
            ax.plot(range(len(means[drug,sentiment])), means[drug,sentiment], 
                    c=color, marker=markers[i%len(markers)], 
                    label=drug.capitalize() if len(drug) > 4 else drug.upper(),
                    fillstyle='none' if i%2==0 else 'full')
            ax.fill_between(range(len(means[drug,sentiment])), 
                            means[drug,sentiment] - stds[drug,sentiment]/2, 
                            means[drug,sentiment] + stds[drug,sentiment]/2, 
                            color=color, alpha=0.05)
        ax.set_title(f'{itags[sentiment].replace("_", " ")} Trajectory of Trip')
        ax.set_xlabel('Time in Narrative')
        ax.set_ylabel(f'{itags[sentiment].replace("_", " ")}')
        ax.set_xticks(())
        ax.set_yticks(())
        ax.legend(ncol=ncol)
    plt.tight_layout()


In [18]:
standardized_d, means_d, stds_d = get_mean_trajectories(scores, column='drug')

['psychoactive_class_int', 'ligand_chemical_int', 'drug_class', 'tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14', 'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21', 'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28', 'tag_29', 'tag_30', 'tag_31', 'tag_32', 'tag_33', 'tag_34', 'tag_35', 'tag_36', 'tag_37', 'tag_38', 'tag_39', 'tag_40', 'tag_41', 'tag_42', 'tag_43', 'tag_44', 'tag_45', 'tag_46', 'tag_47', 'tag_48', 'tag_49', 'tag_50']
vall err 2c-c 65 len: 1 e:x and y arrays must have at least 2 entries
vall err 2c-e 285 len: 1 e:x and y arrays must have at least 2 entries
vall err 2c-p 45 len: 1 e:x and y arrays must have at least 2 entries
vall err 5-meo-tmt 110 len: 1 e:x and y arrays must have at least 2 entries
vall err tma-2 25 len: 1 e:x and y arrays must have at least 2 entries
vall err methamphetamine 420 len: 1 e:x and y arrays must have at least 2 entr

In [19]:
list(tags.keys())

['Small_Group',
 'General',
 'First_Times',
 'Alone',
 'Difficult_Experiences',
 'Glowing_Experiences',
 'Retrospective_Summary',
 'Various',
 'Unknown_Context',
 'Mystical_Experiences',
 'Health_Problems',
 'Combinations',
 'Not_Applicable',
 'Bad_Trips',
 'Hangover_Days_After',
 'Entities_Beings',
 'Music_Discussion',
 'Addiction_Habituation',
 'Post_Trip_Problems',
 'Nature_Outdoors',
 'Relationships',
 'Depression',
 'Therapeutic_Intent_or_Outcome',
 'Overdose',
 'Medical_Use',
 'Sex_Discussion',
 'Train_Wrecks_Trip_Disasters',
 'Guides_Sitters',
 'Rave_Dance_Event',
 'Preparation_Recipes',
 'Festival_Lg_Crowd',
 'Health_Benefits',
 'Large_Group',
 'Multi-Day_Experience',
 'Club_Bar',
 'What_Was_in_That',
 'Personal_Preparation',
 'HPPD_Lasting_Visuals',
 'Families',
 'Second_Hand_Report',
 'Loss_of_Magic',
 'Hospital',
 'Public_Space',
 'School',
 'Poetry',
 'Performance_Enhancement',
 'Large_Party',
 'Group_Ceremony',
 'Workplace',
 'Cultivation_Synthesis',
 'Pregnancy_Baby',
 'M

In [21]:
keep_drugs = list(scores.keys())
keep_drugs = ['5-meo-dmt', 'dmt', 'psilocin', 'mdma',  'hydrocodone', 'sertraline', 'cocaine',
             'venlafaxine', 'methamphetamine', 'hydrocodone', 'morphine', 'oxycodone', 'salvia', 'ketamine',
              'lsd', 'ibogaine', 'cocaine', 'amphetamine',
             '2c-b','2c-i', 'methadone', 'haloperidol']
# keep_drugs = [ 'sertraline',  'paroxetine',
#              'venlafaxine', 'methamphetamine', 'cocaine', 'amphetamine',]

# keep_tags = [
#     'Mystical_Experiences', 'Entities_Beings',
#    # 'Addiction_Habituation', 'Depression' ,  #'Combinations', 'Bad_Trips', 'Various', "Unknown_Context", 'Retrospective_Summary'
    
#             ]
        
        # mdma_tags = [
#              'Rave_Dance_Event', 'Music_Discussion', 'Festival_Lg_Crowd','Club_Bar', 'Large_Group','Relationships', 'Sex_Discussion', 
#     'Glowing_Experiences', 
keep_tags = list(tags.keys()) # ['Rave_Dance_Event']
#keep_tags = ['Rave_Dance_Event', 'Mystical_Experiences', 'Depression' ]
# 'Mystical_Experiences', 'Entities_Beings', 'Depression', 
# plot_tag_trajectories(scores, means, stds, keep_drugs, keep_tags, 
#                       figsize=(9, len(keep_tags)*2), cols=1, rows=len(keep_tags), ncol=6)
plot_tag_trajectories(list(scores.keys()), means_d, stds_d, keep_drugs, keep_tags, 
                      figsize=(9.25, len(keep_tags)*4), cols=1, rows=len(keep_tags), ncol=5)

In [ ]:
standardized_p, means_p, stds_p = get_mean_trajectories(scores)



In [ ]:
keep_tags = list(tags.keys()) # ['Rave_Dance_Event']
keep_drugs = list(scores.keys())
plot_tag_trajectories(df.psychoactive_class.unique(), means_p, stds_p, df.psychoactive_class.unique(), keep_tags, 
                      figsize=(9, len(keep_tags)*4), cols=1, rows=len(keep_tags), ncol=3)

In [ ]:
[c for c in df]

In [ ]:
standardized_c, means_c, stds_c = get_mean_trajectories(scores, column='ligand_chemical_class')

In [ ]:
keep_tags = list(tags.keys()) # ['Rave_Dance_Event']
keep_drugs = list(scores.keys())
plot_tag_trajectories(df.ligand_chemical_class.unique(), means_c, stds_c, 
                      df.ligand_chemical_class.unique(), keep_tags, 
                      figsize=(12, 35*4), cols=1, rows=35, ncol=5)